In [52]:
import os 
from dotenv import load_dotenv, find_dotenv 
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [3]:
from langchain_openai import ChatOpenAI 
llm = ChatOpenAI(model = "gpt-3.5-turbo")

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults 
search = TavilySearchResults()
search.invoke("What is the weather of SF") 

[{'url': 'https://www.weatherapi.com/',
  'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1732410670, 'localtime': '2024-11-23 17:11'}, 'current': {'last_updated_epoch': 1732410000, 'last_updated': '2024-11-23 17:00', 'temp_c': 15.6, 'temp_f': 60.1, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 9.4, 'wind_kph': 15.1, 'wind_degree': 246, 'wind_dir': 'WSW', 'pressure_mb': 1015.0, 'pressure_in': 29.96, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 67, 'cloud': 75, 'feelslike_c': 15.6, 'feelslike_f': 60.1, 'windchill_c': 11.1, 'windchill_f': 52.0, 'heatindex_c': 12.2, 'heatindex_f': 54.0, 'dewpoint_c': 8.4, 'dewpoint_f': 47.2, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 13.5, 'gust_kph': 21.7}}"},
 {'url': 'https://world-weather.info/fo

In [7]:
tools = [search]

In [8]:
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

In [9]:
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [10]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [13]:
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)

In [14]:
from langchain.agents import AgentExecutor 
agent_executor = AgentExecutor(agent = agent, tools = tools, verbose=True)

In [15]:
agent_executor.invoke({"input" : "Hello how are you ?"})



> Entering new AgentExecutor chain...
Hello! I'm here and ready to assist you. How can I help you today?

> Finished chain.


{'input': 'Hello how are you ?',
 'output': "Hello! I'm here and ready to assist you. How can I help you today?"}

In [16]:
agent_executor.invoke({"input": "What is the current whether in Mumbai ?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'current weather in Mumbai'}`


[{'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Mumbai', 'region': 'Maharashtra', 'country': 'India', 'lat': 18.975, 'lon': 72.826, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1732411016, 'localtime': '2024-11-24 06:46'}, 'current': {'last_updated_epoch': 1732410900, 'last_updated': '2024-11-24 06:45', 'temp_c': 24.0, 'temp_f': 75.2, 'is_day': 0, 'condition': {'text': 'Mist', 'icon': '//cdn.weatherapi.com/weather/64x64/night/143.png', 'code': 1030}, 'wind_mph': 5.6, 'wind_kph': 9.0, 'wind_degree': 71, 'wind_dir': 'ENE', 'pressure_mb': 1013.0, 'pressure_in': 29.91, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 69, 'cloud': 0, 'feelslike_c': 25.5, 'feelslike_f': 77.8, 'windchill_c': 24.7, 'windchill_f': 76.4, 'heatindex_c': 26.0, 'heatindex_f': 78.7, 'dewpoint_c': 16.5, 'dewpoint_f': 61.7, 'vis_km': 1.8, 'vis_miles': 1.0, 'uv': 0.0, 

{'input': 'What is the current whether in Mumbai ?',
 'output': 'The current weather in Mumbai is misty with a temperature of 24.0°C (75.2°F). The wind speed is 5.6 mph (9.0 kph) coming from the east-northeast direction. The humidity is at 69%, and the visibility is 1.0 mile.'}

#### **RAG**

In [17]:
from langchain_community.document_loaders import WebBaseLoader 
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [24]:
loader = WebBaseLoader("https://docs.smith.langchain.com/overview")
docs = loader.load()

In [25]:
docs

[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | \uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith', 'description': "LangSmith is a platform for building production-grade LLM applications. It allows you to closely monitor and evaluate your application, so you can ship quickly and with confidence. Use of LangChain's open source frameworks is not necessary - LangSmith works on its own!", 'language': 'en'}, page_content='\n\n\n\n\nGet started with LangSmith | \uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith\n\n\n\n\n\n\nSkip to main contentGo to API DocsSearchRegionUSEUGo to AppQuick StartObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Cloud)AdministrationSelf-hostingReferenceQuick StartOn this pageGet started with LangSmith\nLangSmith is a platform for building production-grade LLM applications. It allows you to closely monitor and evaluate your application, so you can ship quickly and with confidence. Use of LangChain\'s open source fra

In [26]:
documents = RecursiveCharacterTextSplitter(
    chunk_size = 1000, chunk_overlap = 200
).split_documents(docs)

In [27]:
from langchain_openai import OpenAIEmbeddings 
embeddings = OpenAIEmbeddings()

In [29]:
vector = FAISS.from_documents(documents, embeddings)
retriever = vector.as_retriever()

In [34]:
retriever.invoke("What is LangGraph Studio")[0]

Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | \uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith', 'description': "LangSmith is a platform for building production-grade LLM applications. It allows you to closely monitor and evaluate your application, so you can ship quickly and with confidence. Use of LangChain's open source frameworks is not necessary - LangSmith works on its own!", 'language': 'en'}, page_content="Skip to main contentGo to API DocsSearchRegionUSEUGo to AppQuick StartObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Cloud)AdministrationSelf-hostingReferenceQuick StartOn this pageGet started with LangSmith\nLangSmith is a platform for building production-grade LLM applications. It allows you to closely monitor and evaluate your application, so you can ship quickly and with confidence. Use of LangChain's open source frameworks is not necessary - LangSmith works on its own!\n1. Install LangSmith‚Äã\nPythonTy

In [35]:
from langchain.tools.retriever import create_retriever_tool 
retriever_tool = create_retriever_tool(
    retriever, 
    "Langraph_studio", 
    "Search for information about LangGraph. For any questions about LangGraph, you must use this tool!"
)

In [37]:
# Toolkit
tools = [search, retriever_tool]

In [38]:
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(llm, tools, prompt)

In [39]:
from langchain.agents import AgentExecutor 
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [41]:
agent_executor.invoke({"input": "Hi, What is LangGraph studio and how to use it ?"})



> Entering new AgentExecutor chain...

Invoking: `Langraph_studio` with `{'query': 'LangGraph studio'}`


Skip to main contentGo to API DocsSearchRegionUSEUGo to AppQuick StartObservabilityEvaluationPrompt EngineeringDeployment (LangGraph Cloud)AdministrationSelf-hostingReferenceQuick StartOn this pageGet started with LangSmith
LangSmith is a platform for building production-grade LLM applications. It allows you to closely monitor and evaluate your application, so you can ship quickly and with confidence. Use of LangChain's open source frameworks is not necessary - LangSmith works on its own!
1. Install LangSmith‚Äã
PythonTypeScriptpip install -U langsmith openaiyarn add langsmith openai
2. Create an API key‚Äã
To create an API key head to the Settings page. Then click Create API Key.
3. Set up your environment‚Äã
Shellexport LANGCHAIN_TRACING_V2=trueexport LANGCHAIN_API_KEY=<your-api-key># The below examples use the OpenAI API, though it's not necessary in generalexport OPENAI_API_K

{'input': 'Hi, What is LangGraph studio and how to use it ?',
 'output': 'LangGraph Studio is a platform called LangSmith for building production-grade LLM (Large Language Model) applications. It allows you to closely monitor and evaluate your application so you can ship quickly and with confidence. Here\'s how to use LangSmith:\n\n1. **Install LangSmith:**\n   - For Python: `pip install -U langsmith openai`\n   - For TypeScript: `yarn add langsmith openai`\n\n2. **Create an API key:**\n   - Go to the Settings page and click on "Create API Key."\n\n3. **Set up your environment:**\n   - Export the following variables:\n     ```\n     export LANGCHAIN_TRACING_V2=true\n     export LANGCHAIN_API_KEY=<your-api-key>\n     export OPENAI_API_KEY=<your-openai-key>\n     ```\n\n4. **Log your first trace:**\n   - Click the link printed out by your evaluation run to access the LangSmith experiments UI and explore the results of your evaluation.\n\nLangSmith works independently, and you don\'t need

In [43]:
agent_executor.invoke({"input": "What is today's weather in New York"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'New York weather today'}`


[{'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'New York', 'region': 'New York', 'country': 'United States of America', 'lat': 40.7142, 'lon': -74.0064, 'tz_id': 'America/New_York', 'localtime_epoch': 1732411929, 'localtime': '2024-11-23 20:32'}, 'current': {'last_updated_epoch': 1732411800, 'last_updated': '2024-11-23 20:30', 'temp_c': 8.3, 'temp_f': 46.9, 'is_day': 0, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/night/116.png', 'code': 1003}, 'wind_mph': 15.7, 'wind_kph': 25.2, 'wind_degree': 296, 'wind_dir': 'WNW', 'pressure_mb': 1005.0, 'pressure_in': 29.69, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 58, 'cloud': 75, 'feelslike_c': 4.7, 'feelslike_f': 40.5, 'windchill_c': 3.0, 'windchill_f': 37.3, 'heatindex_c': 6.7, 'heatindex_f': 44.0, 'dewpoint_c': 2.2, 'dewpoint_f': 36.0, 'vis_km': 16.0

{'input': "What is today's weather in New York",
 'output': 'The weather in New York today is partly cloudy with a temperature of 46.9°F (8.3°C). The wind speed is 25.2 kph coming from the WNW direction. The humidity is at 58% and the visibility is 9.0 miles.'}

### **ReaAct**

In [53]:
from langchain.utilities import GoogleSerperAPIWrapper 
from langchain.agents import Tool 
from langchain_core.prompts import PromptTemplate 
from langchain.agents import AgentExecutor, create_react_agent 

In [55]:
google_search = GoogleSerperAPIWrapper()

In [56]:
tools = [
    Tool(
        name = "Web Answer", 
        func = google_search.run,
        description="Useful for when you need to ask with search", 
        verbose = True
    )
]

In [57]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [58]:
prompt = PromptTemplate.from_template(template)

In [59]:
search_agent = create_react_agent(llm, tools, prompt)

In [60]:
agent_executor = AgentExecutor(
    agent = search_agent, 
    tools = tools, 
    verbose=True, 
    return_intermediate_steps=True
)

In [61]:
agent_executor.invoke({"input": "What are the spaceX missions ?"})



> Entering new AgentExecutor chain...
I need to find information on the different spaceX missions.
Action: Web Answer
Action Input: "SpaceX missions list"Rockets from the Falcon 9 family have been launched 408 times, with 405 full mission successes, three failures, [a] and one partial failure. Starship's Sixth Flight Test. Learn More. November 18, 2024. NSIL GSAT-N2 Mission. Learn More. November 17, 2024. Starlink Mission. FALCON 1 MAKES HISTORY · DRAGON REACHES THE SPACE STATION · FIRST LAND LANDING · DRONESHIP LANDING · FIRST REFLIGHT · FALCON HEAVY FIRST FLIGHT. List of Falcon 9 and Falcon Heavy launches, SpaceX's current operational launch vehicles. List of SpaceX Starship flight tests. A regularly updated listing of planned orbital missions from spaceports around the globe. Dates and times are given in Greenwich Mean Time. This is a list of all known Falcon 9, Falcon Heavy and Starship launches that SpaceX has planned. The information on this page is based on unofficial ... Spac

{'input': 'What are the spaceX missions ?',
 'output': 'SpaceX missions include launches of Starlink satellites, Dragon crew and cargo missions to the International Space Station, and launches of commercial and military satellites to various orbits.',
 'intermediate_steps': [(AgentAction(tool='Web Answer', tool_input='SpaceX missions list', log='I need to find information on the different spaceX missions.\nAction: Web Answer\nAction Input: "SpaceX missions list"'),
   "Rockets from the Falcon 9 family have been launched 408 times, with 405 full mission successes, three failures, [a] and one partial failure. Starship's Sixth Flight Test. Learn More. November 18, 2024. NSIL GSAT-N2 Mission. Learn More. November 17, 2024. Starlink Mission. FALCON 1 MAKES HISTORY · DRAGON REACHES THE SPACE STATION · FIRST LAND LANDING · DRONESHIP LANDING · FIRST REFLIGHT · FALCON HEAVY FIRST FLIGHT. List of Falcon 9 and Falcon Heavy launches, SpaceX's current operational launch vehicles. List of SpaceX Sta

In [63]:
agent_executor.invoke({"input": "Who is the founder of OpenAI, Who invested it ?"})



> Entering new AgentExecutor chain...
I need to search for information about the founder and investors of OpenAI.
Action: Web Answer
Action Input: "Founder of OpenAI"OpenAI: Artificial intelligence company. OpenAI is an American artificial intelligence research organization founded in December 2015 and headquartered in San Francisco, California. OpenAI Founders: Elon Musk, Sam Altman, Ilya Sutskever, and more. OpenAI CEO: Sam Altman (Nov 29, 2023–). OpenAI CFO: Sarah Friar. OpenAI President: Greg Brockman. OpenAI Founded: December 11, 2015, San Francisco, CA. OpenAI co-founder Wojciech Zaremba stated that he turned down "borderline crazy" offers of two to three times his market value to join OpenAI instead. Samuel Harris Altman (born April 22, 1985) is an American entrepreneur and investor best known as the chief executive officer of OpenAI since 2019. Wojciech Zaremba, a Polish computer scientist, became a co-founder of OpenAI while he was still completing his Ph.D. in deep learning

{'input': 'Who is the founder of OpenAI, Who invested it ?',
 'output': 'The founder of OpenAI is Elon Musk and the investors include Citi, JP Morgan Chase, Thrive Capital, Khosla Ventures, Apple, Nvidia, Microsoft, and others.',
 'intermediate_steps': [(AgentAction(tool='Web Answer', tool_input='Founder of OpenAI', log='I need to search for information about the founder and investors of OpenAI.\nAction: Web Answer\nAction Input: "Founder of OpenAI"'),
   'OpenAI: Artificial intelligence company. OpenAI is an American artificial intelligence research organization founded in December 2015 and headquartered in San Francisco, California. OpenAI Founders: Elon Musk, Sam Altman, Ilya Sutskever, and more. OpenAI CEO: Sam Altman (Nov 29, 2023–). OpenAI CFO: Sarah Friar. OpenAI President: Greg Brockman. OpenAI Founded: December 11, 2015, San Francisco, CA. OpenAI co-founder Wojciech Zaremba stated that he turned down "borderline crazy" offers of two to three times his market value to join Open

### **ReaAct with Custom Tool**

In [67]:
from langchain.tools import tool 
from langchain import hub

In [68]:
# Custom tool for the Agent
@tool 
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
  return fake_employees.get(name,"Employee not found")

In [69]:
# Custom tool for the Agent 
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [70]:
prompt = hub.pull("hwchase17/react")

In [71]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [72]:
tools = [get_employee_salary, get_employee_id]

In [73]:
agent = create_react_agent(llm, tools, prompt)

In [79]:
agent_executor = AgentExecutor(agent = agent,tools = tools, verbose = True)

In [ ]:
agent_executor.invoke({"input" : "What is the salary of Evan ?"})



> Entering new AgentExecutor chain...
I need to first get the employee id of Evan and then use that to get his salary.
Action: get_employee_id
Action Input: "Evan"E005Now that I have Evan's employee id, I can use it to get his salary.
Action: get_employee_salary
Action Input: E00545000I now know the final answer
Final Answer: The salary of Evan is 45000.

> Finished chain.


{'input': 'What is the salary of Evan ?',
 'output': 'The salary of Evan is 45000.'}

In [83]:
agent_executor.invoke({"input" : "What is the salary of Hitesh ?"})



> Entering new AgentExecutor chain...
I need to first get the employee id of Hitesh and then get his salary.
Action: get_employee_id
Action Input: "Hitesh"Employee not foundI need to double check the spelling of the name or see if there are any variations.
Action: get_employee_id
Action Input: "Hitesh Kumar"Employee not foundIt seems like there might be an issue with the database or the employee's name might not be entered correctly.
Action: get_employee_id
Action Input: "Hitesh Sharma"Employee not foundIt appears that Hitesh is not in the database.
Final Answer: The salary of Hitesh is not available as he is not in the database.

> Finished chain.


{'input': 'What is the salary of Hitesh ?',
 'output': 'The salary of Hitesh is not available as he is not in the database.'}

### **Self Ask with Search Agent**

In [92]:
llm = ChatOpenAI(model = "gpt-3.5-turbo")
embedding = OpenAIEmbeddings()

In [94]:
from langchain import hub 
from langchain.agents import AgentExecutor, create_self_ask_with_search_agent 
from langchain_community.tools.tavily_search import TavilyAnswer
from langchain.agents import Tool 

In [98]:
tools = [
    Tool(
        name = "Intermediate Answer", 
        func = google_search.run, 
        description="Useful for when you need to ask with search", 
        verbose = True
    )
]

In [99]:
prompt = hub.pull("hwchase17/self-ask-with-search")

In [102]:
print(prompt.template)

Question: Who lived longer, Muhammad Ali or Alan Turing?
Are follow up questions needed here: Yes.
Follow up: How old was Muhammad Ali when he died?
Intermediate answer: Muhammad Ali was 74 years old when he died.
Follow up: How old was Alan Turing when he died?
Intermediate answer: Alan Turing was 41 years old when he died.
So the final answer is: Muhammad Ali

Question: When was the founder of craigslist born?
Are follow up questions needed here: Yes.
Follow up: Who was the founder of craigslist?
Intermediate answer: Craigslist was founded by Craig Newmark.
Follow up: When was Craig Newmark born?
Intermediate answer: Craig Newmark was born on December 6, 1952.
So the final answer is: December 6, 1952

Question: Who was the maternal grandfather of George Washington?
Are follow up questions needed here: Yes.
Follow up: Who was the mother of George Washington?
Intermediate answer: The mother of George Washington was Mary Ball Washington.
Follow up: Who was the father of Mary Ball Washin

In [100]:
agent = create_self_ask_with_search_agent(llm, tools, prompt)

In [103]:
agent_executor = AgentExecutor(agent = agent, tools = tools, verbose=True, handle_parsing_errors=True)

In [104]:
agent_executor.invoke({"input": "Where is a hometown of the current prime minister of india ?"})



> Entering new AgentExecutor chain...
Could not parse output: No.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE Invalid or incomplete responseIt seems there was an issue with the response. Let me try again.

Question: Where is the hometown of the current prime minister of India?
Are follow up questions needed here: Yes.
Follow up: Who is the current prime minister of India?Narendra ModiFollow up: What is the hometown of Narendra Modi?Modi was born and raised in Vadnagar in northeastern Gujarat, where he completed his secondary education. He was introduced to the RSS at the age of eight. At ... Narendra Modi (born September 17, 1950, Vadnagar, India) is an Indian politician and government official who rose to become a senior leader ... Narendra Modi's inspiring life journey to the Office of Prime Minister began in the by-lanes of Vadnagar, a small town in North Gujarat's Mehsana district. He ... Place of Birth. Vadnagar, Ma

{'input': 'Where is a hometown of the current prime minister of india ?',
 'output': 'Vadnagar, Gujarat'}

In [107]:
agent_executor.invoke({"input": "Which asian country have most billionaires in particular city and how many there are billionaires in that city ?"})



> Entering new AgentExecutor chain...
Could not parse output: Yes.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE Invalid or incomplete responseFollow up: Which city has the most billionaires in Asia?India's financial capital Mumbai emerged as the “billionaire capital of Asia” with more billionaires than China's capital Beijing, as per the 2024 Hurun India Rich List. Mumbai saw an increase of 58 billionaire individuals, taking its total to 386 on the list.So the final answer is: Mumbai, India with 386 billionaires

> Finished chain.


{'input': 'Which asian country have most billionaires in particular city and how many there are billionaires in that city ?',
 'output': 'Mumbai, India with 386 billionaires'}